# 
In Notebook 02, you learned how Deployments keep pods running. In this notebook, you will learn how those pods talk to each other and how traffic enters the cluster. Think of this notebook as the road system for your microservices: Services are the stable roads inside the cluster, and Ingress is the front door from the outside world.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain why Kubernetes Services exist and why pod IP addresses are not enough
- Create and test a **ClusterIP** Service for internal communication
- Create and test a **NodePort** Service for local external access in minikube
- Understand Kubernetes DNS names such as `user-service.k8s-lab.svc.cluster.local`
- Apply the shared Service manifests for `api-gateway`, `user-service`, and `order-service`
- Test communication between microservices inside the `k8s-lab` namespace
- Enable the NGINX Ingress Controller in minikube and route HTTP traffic through an Ingress resource
- Clean up the networking resources you created


## 
       Before starting, make sure your local lab environment is ready.

       1. Open this notebook from the `kubernetes/notebooks/` folder.
'Reload Window'.
       3. Confirm that minikube is running and that the apps from Notebook 02 are already deployed in the `k8s-lab` namespace.
       4. If Notebook 02 is not finished yet, go back and deploy the three sample FastAPI apps first.

       The commands below check the cluster, namespace, deployments, and pods.


In [ ]:
!kubectl config current-context
!kubectl get namespace k8s-lab
!kubectl get deploy,pods -n k8s-lab -o wide


## 
A pod is **ephemeral**, which means Kubernetes is free to replace it at any time. If a node fails or you roll out a new version, the old pod can disappear and a new pod will be created with a different IP address.

That creates a problem: if `api-gateway` tries to call `user-service` by a pod IP, that connection can break as soon as Kubernetes replaces the pod.

A **Service** solves this by giving a workload a **stable name** and a **stable virtual IP**. The pods behind the Service can come and go, but the Service name stays the same. Your applications talk to the Service, and Kubernetes forwards traffic to healthy pods behind it.

A good mental model is this:

- **Pods** are the workers
- **Services** are the stable phone numbers for those workers
- **Ingress** is the public receptionist at the front desk


In [ ]:
!kubectl get pods -n k8s-lab -o wide

# Exercise: look at the pod IPs above.
# If one of these pods is recreated, its IP can change.
# That is exactly why we use a Service instead of hard-coding pod addresses.


## 
Here is the basic idea behind a Kubernetes Service. One stable Service sits in front of multiple pods. Kubernetes load balances requests across the healthy pods for you.

```text
+-----------+        +-------------------+        +-----------------+
| External  | -----> | Service           | -----> | Pod 1           |
| Request   |        | user-service      | -----> | Pod 2           |
| or Client |        | Stable virtual IP | -----> | Pod 3           |
+-----------+        +-------------------+        +-----------------+
```

The client does **not** need to know which exact pod handled the request. It only needs to know the Service name.


In [ ]:
!kubectl get deploy user-service -n k8s-lab
!kubectl get pods -l app=user-service -n k8s-lab -o wide

# Exercise: count how many user-service pods exist.
# A Service can sit in front of one pod or many pods.


## 
`ClusterIP` is the default Service type in Kubernetes. It creates an internal virtual IP that other pods can reach, but it does **not** expose the app outside the cluster.

This is perfect for internal microservice-to-microservice communication. In our lab, `user-service` should be reachable from other workloads inside the cluster, but it does not need to be public.

In the next cell, you will create a `ClusterIP` Service named `user-service` and then inspect it.


In [ ]:
!kubectl expose deployment user-service --name user-service --type=ClusterIP --port=8001 --target-port=8001 -n k8s-lab --dry-run=client -o yaml | kubectl apply -f -
!kubectl get svc user-service -n k8s-lab
!kubectl describe svc user-service -n k8s-lab


## 
A Service is only useful if other workloads can reach it. The easiest way to test this is to launch a temporary helper pod that already contains `curl`.

The command below starts a short-lived pod, sends an HTTP request to `user-service`, prints the response, and then removes the helper pod automatically.


In [ ]:
!kubectl run curl-demo --rm --restart=Never --image=curlimages/curl:8.7.1 -n k8s-lab --command -- curl -s http://user-service:8001/health


## 
A `NodePort` Service opens a port on the Kubernetes node. In minikube, this is a simple way to access a service from your laptop.

For this lab, we will expose `api-gateway` with a `NodePort` Service. That gives us a stable entry point for local testing before we move to Ingress.

This is useful for learning because you can see the app from outside the cluster without needing a cloud load balancer.


In [ ]:
!kubectl expose deployment api-gateway --name api-gateway --type=NodePort --port=8000 --target-port=8000 -n k8s-lab --dry-run=client -o yaml | kubectl apply -f -
!kubectl get svc api-gateway -n k8s-lab
!minikube service api-gateway -n k8s-lab --url


## 
Kubernetes includes built-in service discovery through DNS. When you create a Service, Kubernetes automatically gives it a DNS name.

The full DNS name pattern looks like this:

`service-name.namespace.svc.cluster.local`

For our user service, that becomes:

`user-service.k8s-lab.svc.cluster.local`

The nice part is that inside the same namespace, you can usually use the short name like `user-service`. Kubernetes expands it for you behind the scenes.

That means your applications can use stable hostnames instead of unstable pod IPs.


In [ ]:
!kubectl run dns-test --rm --restart=Never --image=busybox:1.36 -n k8s-lab -- nslookup user-service.k8s-lab.svc.cluster.local

# Exercise: run the same command again with just 'user-service' to see the short name also works inside the namespace.


## 
So far, you created `user-service` and `api-gateway` one at a time so you could see each idea clearly. Now let's apply the shared lab manifest that defines all three Services together:

- `api-gateway` on port 8000 as a `NodePort`
- `user-service` on port 8001 as a `ClusterIP`
- `order-service` on port 8002 as a `ClusterIP`

This manifest lives in `../manifests/service.yaml` relative to this notebook.


In [ ]:
!kubectl apply -f ../manifests/service.yaml
!kubectl get svc -n k8s-lab


## 
Now let's prove that one service can call another service inside the cluster.

The `api-gateway` deployment is already configured with the service URLs:

- `http://user-service:8001`
- `http://order-service:8002`

In the next cell, you will execute a command inside the `api-gateway` container and send a request to `user-service:8001/health`.

If `curl` is missing in the image, the command falls back to `wget` so the exercise is still beginner-friendly.


In [ ]:
!kubectl exec -n k8s-lab deploy/api-gateway -- sh -c "curl -s http://user-service:8001/health || wget -qO- http://user-service:8001/health"


## 
A `NodePort` is great for learning, but it is a little low-level. In real Kubernetes setups, HTTP traffic usually enters through an **Ingress Controller**.

An Ingress Controller watches `Ingress` resources and turns rules into working HTTP routing. In minikube, the easiest option is the built-in NGINX ingress addon.

Think of the Ingress Controller as the traffic manager at the cluster entrance. It can decide which Service should receive a request based on the path or hostname.


In [ ]:
!minikube addons enable ingress
!kubectl get pods -n ingress-nginx


## 
We want requests arriving at the cluster to be sent to the `api-gateway` Service. To keep testing simple, this Ingress uses a path rule without a host name, so you can test it directly with the minikube IP.

```text
+-------------+      +-------------------+      +-----------------+
| Browser or  | ---> | NGINX Ingress     | ---> | api-gateway     |
| curl client |      | path-based router |      | Service         |
+-------------+      +-------------------+      +-----------------+
```

The first cell writes the YAML file. The second cell applies it to the cluster.


In [ ]:
%%writefile ./api-gateway-ingress.yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: api-gateway-ingress
  namespace: k8s-lab
spec:
  ingressClassName: nginx
  rules:
    - http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: api-gateway
                port:
                  number: 8000


In [ ]:
!kubectl apply -f ./api-gateway-ingress.yaml
!kubectl get ingress -n k8s-lab
!kubectl describe ingress api-gateway-ingress -n k8s-lab


## 
Now you can send traffic to the minikube IP. The Ingress controller receives the request and forwards it to `api-gateway`, which then handles the `/api/users` route.

If your app returns a JSON response, that means the full traffic path is working:

`client -> ingress -> service -> pod`


In [ ]:
!curl http://$(minikube ip)/api/users


## 
Use the next cell when you want to remove the Ingress resource created in this notebook. We leave the shared Services in place because later notebooks may still use them.

If you want to keep experimenting, you can skip this cleanup cell for now and run it later.


In [ ]:
!kubectl delete ingress api-gateway-ingress -n k8s-lab --ignore-not-found
!kubectl delete -f ./api-gateway-ingress.yaml --ignore-not-found


## 
Great work. In this notebook, you learned that:

- Pods are temporary, so they are not a safe networking endpoint by themselves
- A **Service** gives your app a stable name and stable virtual IP
- **ClusterIP** is best for internal communication inside the cluster
- **NodePort** is a simple way to expose an app from minikube to your laptop
- Kubernetes DNS lets pods discover Services by name
- The sample apps in `k8s-lab` can talk to each other through Service names like `user-service`
- An **Ingress Controller** provides a friendlier HTTP entry point than raw node ports
- An **Ingress** resource lets you route requests from outside the cluster to the correct Service

If Notebook 03 made sense, you are ready for Notebook 04 where you will package Kubernetes apps with Helm and customize them with Kustomize.
